In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
from typing import Optional
from pathlib import Path
import json
import shutil

In [3]:
def make_input_folder(
    model_name_and_path: Optional[tuple[str, Path]],
    inputs: list[tuple[Path, str]],
    dest: Path,
):
    dest.mkdir(exist_ok=True, parents=True)
    meta = {}

    # save model first
    if model_name_and_path is not None:
        name, path = model_name_and_path
        shutil.copy(path, dest / path.name)
        meta["model"] = {
            "name": name,
            "path": path.name,
        }

    meta["inputs"] = []
    inputs_dir = dest / "inputs"
    inputs_dir.mkdir(parents=True, exist_ok=True)
    for input_path, label in inputs:
        dest_input_path = inputs_dir / input_path.name
        shutil.copy(input_path, dest_input_path)
        meta["inputs"].append(
            {
                "path": str(Path("inputs") / input_path.name),
                "label": label,
            }
        )
    
    with open(dest / "meta.json", "w") as f:
        json.dump(meta, f)

In [4]:
from pt_to_api.mnist import get_learner, get_mnist_dataloader
learner = get_learner()

In [5]:
dl = get_mnist_dataloader(1, bs=4)
len(dl.train.items)

60000

In [6]:
import random
import string

def get_random_string(length):
    # Choose from letters (a-z, A-Z) and digits (0-9)
    characters = string.ascii_letters + string.digits
    return ''.join(random.choices(characters, k=length))

In [7]:
import torch

def make_examples_from_dl(dloader, allowed_labels=None, max_samples=20):
    inputs_and_labels = []
    for batch, labels in dloader:
        for i in range(len(batch)):
            if len(inputs_and_labels) >= max_samples:
                break
            inp_tens, label = batch[i], labels[i].item()
            if allowed_labels is not None:
                if label not in allowed_labels:
                    continue
            inp_tens = inp_tens.unsqueeze(0)
            inputs_and_labels.append((inp_tens, label))
        if len(inputs_and_labels) >= max_samples:
            break
    return inputs_and_labels

def dump_examples_in_folder(
    tmp: Path, examples: list[tuple[torch.Tensor, int]]
):
    paths_and_labels = []
    for inp_tens, label in examples:
        fname = f"{label}-{get_random_string(3)}.pt"
        dest = tmp / fname
        torch.save(inp_tens, dest)
        paths_and_labels.append((dest, label))
    return paths_and_labels

## Verify

In [ ]:
# verify
import matplotlib.pyplot as plt
from pt_to_api.utils import show_single_channel_red_green_black
import shutil


tmp = Path("./tmp")
shutil.rmtree(tmp, True)
tmp.mkdir(exist_ok=True, parents=True)
examples = make_examples_from_dl(dl.train, max_samples=8)
paths_and_labels = dump_examples_in_folder(tmp, examples)

r = []
for p, l in paths_and_labels:
    t = torch.load(p, "cpu", weights_only=False)
    r.append(t[0][0])
_ = show_single_channel_red_green_black(r, 20, ncols=8)
plt.show()

shutil.rmtree(tmp, True)

## Dump

In [10]:
from collections import defaultdict
import numpy as np
import random
import torch
import tempfile

ORIG_MODEL_PT_FILE = Path("/Users/hariomnarang/Desktop/personal/hiccup-ide/backend/test_load/model.pt")
MODEL_NAME = "simple_mnist_v1"
DEST = Path("../folder_with_all_inputs")
shutil.rmtree(DEST, True)
DEST.mkdir(exist_ok=True, parents=True)
CATS = [4]
N_SAMPLES = 5000

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    examples = make_examples_from_dl(dl.train, CATS, N_SAMPLES)
    paths_and_labels = dump_examples_in_folder(tmp, examples)
    print("examples length", len(paths_and_labels))
    model_name_and_path = (MODEL_NAME, ORIG_MODEL_PT_FILE)
    make_input_folder(model_name_and_path, paths_and_labels, DEST)

examples length 5000


In [11]:
# ive already done a 100 4s. hmmmm, lets do 9 then

In [11]:
! cd {DEST.resolve()} && py_tree 10 2

.
|-- inputs
|   |-- 4-003.pt
|   |-- 4-00C.pt
|   `-- ... (4699 more files)
|-- meta.json
`-- model.pt


In [15]:
! cd ../hiccup_ide && uv run manage.py load_folder {DEST.resolve()}

Processing model: simple_mnist_v1 from /Users/hariomnarang/Desktop/personal/hiccup-ide/backend/folder_with_all_inputs/model.pt
Loading weights for model simple_mnist_v1...
Loaded 160 weights
Processing input: 4-Jqq (label: 4)
WARN: skipping input: 4-Jqq, already exists
done: 0
Processing input: 4-mlx (label: 4)
WARN: skipping input: 4-mlx, already exists
Processing input: 4-UBB (label: 4)
WARN: skipping input: 4-UBB, already exists
Processing input: 4-12v (label: 4)
WARN: skipping input: 4-12v, already exists
Processing input: 4-8tD (label: 4)
WARN: skipping input: 4-8tD, already exists
Processing input: 4-ZIs (label: 4)
WARN: skipping input: 4-ZIs, already exists
Processing input: 4-gPc (label: 4)
WARN: skipping input: 4-gPc, already exists
Processing input: 4-nSQ (label: 4)
WARN: skipping input: 4-nSQ, already exists
Processing input: 4-6Yh (label: 4)
WARN: skipping input: 4-6Yh, already exists
Processing input: 4-Oow (label: 4)
WARN: skipping input: 4-Oow, already exists
Processing 